In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

MEN_PATH = RAW_DIR / "MRegularSeasonCompactResults.csv"
WOMEN_PATH = RAW_DIR / "WRegularSeasonCompactResults.csv"

In [2]:
men_games = pd.read_csv(MEN_PATH)
women_games = pd.read_csv(WOMEN_PATH)

print("Men shape:", men_games.shape)
print("Women shape:", women_games.shape)

display(men_games.head())
display(women_games.head())

Men shape: (198577, 8)
Women shape: (142507, 8)


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1998,18,3104,91,3202,41,H,0
1,1998,18,3163,87,3221,76,H,0
2,1998,18,3222,66,3261,59,H,0
3,1998,18,3307,69,3365,62,H,0
4,1998,18,3349,115,3411,35,H,0


In [3]:
def inspect_regular_season_data(df, name):
    print(f"\n{name}")
    print("-" * 40)

    print("Season range:", df["Season"].min(), "-", df["Season"].max())
    print("Number of games:", len(df))
    print("Missing values:")
    print(df[["Season", "DayNum", "WTeamID", "LTeamID", "WLoc"]].isna().sum())

    print("\nWLoc values:")
    print(df["WLoc"].value_counts(dropna=False).sort_index())


inspect_regular_season_data(men_games, "MEN")
inspect_regular_season_data(women_games, "WOMEN")


MEN
----------------------------------------
Season range: 1985 - 2026
Number of games: 198577
Missing values:
Season     0
DayNum     0
WTeamID    0
LTeamID    0
WLoc       0
dtype: int64

WLoc values:
WLoc
A     61033
H    117154
N     20390
Name: count, dtype: int64

WOMEN
----------------------------------------
Season range: 1998 - 2026
Number of games: 142507
Missing values:
Season     0
DayNum     0
WTeamID    0
LTeamID    0
WLoc       0
dtype: int64

WLoc values:
WLoc
A    51233
H    80544
N    10730
Name: count, dtype: int64


In [4]:
men_games = (
    men_games
    .sort_values(["Season", "DayNum"])
    .reset_index(drop=True)
)

women_games = (
    women_games
    .sort_values(["Season", "DayNum"])
    .reset_index(drop=True)
)

print("Men:")
display(men_games[["Season", "DayNum", "WTeamID", "LTeamID", "WLoc"]].head(10))

print("Women:")
display(women_games[["Season", "DayNum", "WTeamID", "LTeamID", "WLoc"]].head(10))

Men:


,Season,DayNum,WTeamID,LTeamID,WLoc
0,1985,20,1228,1328,N
1,1985,25,1106,1354,H
2,1985,25,1112,1223,H
3,1985,25,1165,1432,H
4,1985,25,1192,1447,H
5,1985,25,1218,1337,H
6,1985,25,1228,1226,N
7,1985,25,1242,1268,N
8,1985,25,1260,1133,H
9,1985,25,1305,1424,H


Women:


,Season,DayNum,WTeamID,LTeamID,WLoc
0,1998,18,3104,3202,H
1,1998,18,3163,3221,H
2,1998,18,3222,3261,H
3,1998,18,3307,3365,H
4,1998,18,3349,3411,H
5,1998,18,3435,3172,H
6,1998,18,3443,3257,H
7,1998,19,3119,3217,H
8,1998,19,3139,3103,H
9,1998,19,3141,3387,H


In [5]:
def expected_score(rating_a, rating_b, scale=400):
    """
    Probability that Team A beats Team B based on Elo ratings.
    """
    return 1 / (1 + 10 ** ((rating_b - rating_a) / scale))

In [6]:
print("Equal teams:")
print(expected_score(1500, 1500))

print("\nA is 100 Elo stronger:")
print(expected_score(1600, 1500))

print("\nA is 100 Elo weaker:")
print(expected_score(1400, 1500))

print("\nA is 400 Elo stronger:")
print(expected_score(1900, 1500))

Equal teams:
0.5

A is 100 Elo stronger:
0.6400649998028851

A is 100 Elo weaker:
0.35993500019711494

A is 400 Elo stronger:
0.9090909090909091


In [9]:
def run_elo_season(
    season_games,
    k_factor,
    initial_elo=1500,
    scale=400,
    home_advantage=60
):
    """
    Run Elo sequentially through one regular season.

    Each game is stored from the lower-TeamID (Team1) perspective,
    matching the representation used by the tournament model.

    Returns:
        predictions_df:
            Pre-game Elo probabilities, targets and Brier errors.

        final_ratings:
            Final Elo rating for every team after the season.
    """

    ratings = {}
    predictions = []

    season_games = (
        season_games
        .sort_values(["DayNum"])
        .reset_index(drop=True)
    )

    for _, game in season_games.iterrows():

        winner = int(game["WTeamID"])
        loser = int(game["LTeamID"])
        location = game["WLoc"]

        winner_elo = ratings.get(winner, initial_elo)
        loser_elo = ratings.get(loser, initial_elo)

        # Temporary home-court adjustment for prediction only.
        if location == "H":
            winner_adjusted = winner_elo + home_advantage
            loser_adjusted = loser_elo

        elif location == "A":
            winner_adjusted = winner_elo
            loser_adjusted = loser_elo + home_advantage

        else:
            winner_adjusted = winner_elo
            loser_adjusted = loser_elo

        # Probability calculated BEFORE Elo is updated.
        winner_prob = expected_score(
            winner_adjusted,
            loser_adjusted,
            scale=scale
        )

        # Fixed matchup representation:
        # Team1 is always the lower TeamID.
        team1 = min(winner, loser)
        team2 = max(winner, loser)

        if team1 == winner:
            team1_prob = winner_prob
            target = 1
        else:
            team1_prob = 1 - winner_prob
            target = 0

        brier = (team1_prob - target) ** 2

        predictions.append({
            "Season": int(game["Season"]),
            "DayNum": int(game["DayNum"]),
            "Team1": team1,
            "Team2": team2,
            "Team1Prob": team1_prob,
            "Target": target,
            "Brier": brier
        })

        # Elo update occurs only after prediction.
        winner_change = k_factor * (1 - winner_prob)

        ratings[winner] = winner_elo + winner_change
        ratings[loser] = loser_elo - winner_change

    predictions_df = pd.DataFrame(predictions)

    return predictions_df, ratings

In [10]:
men_2003 = men_games[men_games["Season"] == 2003].copy()

test_predictions, test_final_ratings = run_elo_season(
    men_2003,
    k_factor=80
)

print("Number of games:", len(test_predictions))
print("Number of teams rated:", len(test_final_ratings))

display(test_predictions.head(10))

print("Mean Brier:", test_predictions["Brier"].mean())

Number of games: 4616
Number of teams rated: 327


,Season,DayNum,Team1,Team2,Team1Prob,Target,Brier
0,2003,10,1104,1328,0.500000,1,0.250000
1,2003,10,1272,1393,0.500000,1,0.250000
2,2003,11,1266,1437,0.500000,1,0.250000
3,2003,11,1296,1457,0.500000,1,0.250000
4,2003,11,1208,1400,0.500000,0,0.250000
5,2003,11,1186,1458,0.414501,0,0.171811
6,2003,12,1161,1236,0.585499,1,0.171811
7,2003,12,1186,1457,0.509842,1,0.240255
8,2003,12,1156,1194,0.500000,0,0.250000
9,2003,12,1296,1458,0.424088,0,0.179851


Mean Brier: 0.19582572453881247


In [11]:
K_VALUES = [20, 40, 60, 80, 100]

MEN_DEV_YEARS = list(range(2003, 2017))
MEN_VAL_YEARS = list(range(2017, 2022))

WOMEN_DEV_YEARS = list(range(2010, 2017))
WOMEN_VAL_YEARS = list(range(2017, 2022))

print("Men development:", MEN_DEV_YEARS)
print("Men validation:", MEN_VAL_YEARS)

print("\nWomen development:", WOMEN_DEV_YEARS)
print("Women validation:", WOMEN_VAL_YEARS)

Men development: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
Men validation: [2017, 2018, 2019, 2020, 2021]

Women development: [2010, 2011, 2012, 2013, 2014, 2015, 2016]
Women validation: [2017, 2018, 2019, 2020, 2021]


In [12]:
def evaluate_k_by_season(
    games,
    k_factor,
    years,
    initial_elo=1500,
    scale=400,
    home_advantage=60
):
    """
    Evaluate one K-factor independently across multiple seasons.

    Since our Elo methodology resets every team to 1500 at the
    beginning of each season, each season is run separately.
    """

    results = []

    for year in years:

        season_games = games[games["Season"] == year].copy()

        predictions, _ = run_elo_season(
            season_games,
            k_factor=k_factor,
            initial_elo=initial_elo,
            scale=scale,
            home_advantage=home_advantage
        )

        results.append({
            "Season": year,
            "K": k_factor,
            "Games": len(predictions),
            "Brier": predictions["Brier"].mean()
        })

    return pd.DataFrame(results)

In [13]:
def run_k_experiment(games, development_years, validation_years):

    development_results = []
    validation_results = []

    for k in K_VALUES:

        dev = evaluate_k_by_season(
            games,
            k_factor=k,
            years=development_years
        )

        val = evaluate_k_by_season(
            games,
            k_factor=k,
            years=validation_years
        )

        development_results.append(dev)
        validation_results.append(val)

    development_results = pd.concat(
        development_results,
        ignore_index=True
    )

    validation_results = pd.concat(
        validation_results,
        ignore_index=True
    )

    return development_results, validation_results

In [14]:
men_dev_results, men_val_results = run_k_experiment(
    men_games,
    MEN_DEV_YEARS,
    MEN_VAL_YEARS
)

men_dev_summary = (
    men_dev_results
    .groupby("K")
    .agg(
        DevelopmentMeanBrier=("Brier", "mean"),
        DevelopmentStdBrier=("Brier", "std")
    )
)

men_val_table = (
    men_val_results
    .pivot(
        index="K",
        columns="Season",
        values="Brier"
    )
)

men_val_table["ValidationMeanBrier"] = (
    men_val_table.mean(axis=1)
)

men_val_table["ValidationStdBrier"] = (
    men_val_table[MEN_VAL_YEARS].std(axis=1)
)

men_summary = men_dev_summary.join(men_val_table)

display(
    men_summary.sort_values("ValidationMeanBrier")
)

,DevelopmentMeanBrier,DevelopmentStdBrier,2017,2018,2019,2020,2021,ValidationMeanBrier,ValidationStdBrier
K,,,,,,,,,
60,0.193953,0.002742,0.197153,0.196098,0.197992,0.199744,0.209511,0.200100,0.005428
80,0.193759,0.003082,0.196874,0.196299,0.198149,0.200057,0.209567,0.200189,0.005437
100,0.194983,0.003359,0.197987,0.197825,0.199691,0.201641,0.210964,0.201622,0.005446
40,0.196827,0.002298,0.200028,0.198458,0.200451,0.201819,0.211841,0.202519,0.005347
20,0.205725,0.001659,0.208732,0.206677,0.208784,0.209271,0.219159,0.210524,0.004929


In [15]:
women_dev_results, women_val_results = run_k_experiment(
    women_games,
    WOMEN_DEV_YEARS,
    WOMEN_VAL_YEARS
)

women_dev_summary = (
    women_dev_results
    .groupby("K")
    .agg(
        DevelopmentMeanBrier=("Brier", "mean"),
        DevelopmentStdBrier=("Brier", "std")
    )
)

women_val_table = (
    women_val_results
    .pivot(
        index="K",
        columns="Season",
        values="Brier"
    )
)

women_val_table["ValidationMeanBrier"] = (
    women_val_table.mean(axis=1)
)

women_val_table["ValidationStdBrier"] = (
    women_val_table[WOMEN_VAL_YEARS].std(axis=1)
)

women_summary = women_dev_summary.join(women_val_table)

display(
    women_summary.sort_values("ValidationMeanBrier")
)

,DevelopmentMeanBrier,DevelopmentStdBrier,2017,2018,2019,2020,2021,ValidationMeanBrier,ValidationStdBrier
K,,,,,,,,,
100,0.184654,0.003962,0.182910,0.179458,0.181803,0.185695,0.195663,0.185106,0.006313
80,0.184905,0.003596,0.183423,0.180250,0.182551,0.186285,0.196243,0.185750,0.006251
60,0.186926,0.003164,0.185663,0.182979,0.185143,0.188658,0.198619,0.188213,0.006160
40,0.192264,0.002610,0.191168,0.189303,0.191175,0.194335,0.204250,0.194046,0.005984
20,0.205053,0.001817,0.204100,0.203623,0.204881,0.207324,0.216709,0.207327,0.005434


In [16]:
WOMEN_REFINED_K = [80, 100, 120, 140]

women_refined_results = []

for k in WOMEN_REFINED_K:

    result = evaluate_k_by_season(
        women_games,
        k_factor=k,
        years=WOMEN_VAL_YEARS
    )

    women_refined_results.append(result)

women_refined_results = pd.concat(
    women_refined_results,
    ignore_index=True
)

women_refined_table = (
    women_refined_results
    .pivot(
        index="K",
        columns="Season",
        values="Brier"
    )
)

women_refined_table["ValidationMeanBrier"] = (
    women_refined_table[WOMEN_VAL_YEARS].mean(axis=1)
)

women_refined_table["ValidationStdBrier"] = (
    women_refined_table[WOMEN_VAL_YEARS].std(axis=1)
)

display(
    women_refined_table.sort_values("ValidationMeanBrier")
)

Season,2017,2018,2019,2020,2021,ValidationMeanBrier,ValidationStdBrier
K,,,,,,,
100,0.182910,0.179458,0.181803,0.185695,0.195663,0.185106,0.006313
120,0.183435,0.179838,0.182166,0.186181,0.196185,0.185561,0.006365
80,0.183423,0.180250,0.182551,0.186285,0.196243,0.185750,0.006251
140,0.184639,0.180984,0.183251,0.187368,0.197441,0.186737,0.006415


In [17]:
MEN_FINAL_K = 60
WOMEN_FINAL_K = 100

INITIAL_ELO = 1500
ELO_SCALE = 400
HOME_ADVANTAGE = 60

print("Final men's K:", MEN_FINAL_K)
print("Final women's K:", WOMEN_FINAL_K)

Final men's K: 60
Final women's K: 100


In [18]:
def generate_final_elo_ratings(
    games,
    k_factor,
    initial_elo=1500,
    scale=400,
    home_advantage=60
):
    """
    Generate end-of-regular-season Elo ratings for every team-season.

    Ratings reset to initial_elo at the beginning of each season.
    """

    all_ratings = []

    for season in sorted(games["Season"].unique()):

        season_games = games[
            games["Season"] == season
        ].copy()

        _, final_ratings = run_elo_season(
            season_games,
            k_factor=k_factor,
            initial_elo=initial_elo,
            scale=scale,
            home_advantage=home_advantage
        )

        for team_id, final_elo in final_ratings.items():

            all_ratings.append({
                "Season": int(season),
                "TeamID": int(team_id),
                "FinalElo": final_elo
            })

    return pd.DataFrame(all_ratings)

In [19]:
men_elo = generate_final_elo_ratings(
    men_games,
    k_factor=MEN_FINAL_K,
    initial_elo=INITIAL_ELO,
    scale=ELO_SCALE,
    home_advantage=HOME_ADVANTAGE
)

women_elo = generate_final_elo_ratings(
    women_games,
    k_factor=WOMEN_FINAL_K,
    initial_elo=INITIAL_ELO,
    scale=ELO_SCALE,
    home_advantage=HOME_ADVANTAGE
)

print("Men Elo shape:", men_elo.shape)
print("Women Elo shape:", women_elo.shape)

display(men_elo.head())
display(women_elo.head())

Men Elo shape: (13753, 3)
Women Elo shape: (9851, 3)


,Season,TeamID,FinalElo
0,1985,1228,1748.562289
1,1985,1328,1840.722709
2,1985,1417,1638.757980
3,1985,1225,1265.971331
4,1985,1412,1680.784833


,Season,TeamID,FinalElo
0,1998,3104,1916.856967
1,1998,3202,1521.946455
2,1998,3163,2000.588851
3,1998,3221,1775.678821
4,1998,3222,1347.543298


In [20]:
def validate_elo_output(df, name):

    print(f"\n{name}")
    print("-" * 40)

    print(
        "Season range:",
        df["Season"].min(),
        "-",
        df["Season"].max()
    )

    print("Rows:", len(df))

    print(
        "Duplicate Season-TeamID:",
        df.duplicated(["Season", "TeamID"]).sum()
    )

    print(
        "Missing values:",
        df.isna().sum().sum()
    )

    print(
        "\nFinalElo summary:"
    )

    display(df["FinalElo"].describe())


validate_elo_output(men_elo, "MEN")
validate_elo_output(women_elo, "WOMEN")


MEN
----------------------------------------
Season range: 1985 - 2026
Rows: 13753
Duplicate Season-TeamID: 0
Missing values: 0

FinalElo summary:


count    13753.000000
mean      1500.000000
std        160.120394
min       1026.787684
25%       1385.013897
50%       1495.486232
75%       1613.654933
max       2000.726332
Name: FinalElo, dtype: float64


WOMEN
----------------------------------------
Season range: 1998 - 2026
Rows: 9851
Duplicate Season-TeamID: 0
Missing values: 0

FinalElo summary:


count    9851.000000
mean     1500.000000
std       234.072422
min       868.914695
25%      1331.429796
50%      1491.124980
75%      1662.532704
max      2247.584665
Name: FinalElo, dtype: float64

In [21]:
def final_elo_checks(df, name):

    print(f"\n{name}")
    print("=" * 50)

    # Mean Elo by season should remain approximately 1500
    season_means = (
        df.groupby("Season")["FinalElo"]
        .mean()
    )

    print("Season mean Elo range:")
    print(
        season_means.min(),
        "-",
        season_means.max()
    )

    # Inspect latest season
    latest = df[df["Season"] == 2026]

    print("\n2026 teams:", len(latest))

    print("\n2026 Elo summary:")
    display(latest["FinalElo"].describe())

    print("\nHighest 10 Elo ratings in 2026:")
    display(
        latest
        .sort_values("FinalElo", ascending=False)
        .head(10)
    )

    print("\nLowest 10 Elo ratings in 2026:")
    display(
        latest
        .sort_values("FinalElo")
        .head(10)
    )


final_elo_checks(men_elo, "MEN")
final_elo_checks(women_elo, "WOMEN")


MEN
Season mean Elo range:
1500.0 - 1500.0

2026 teams: 365

2026 Elo summary:


count     365.000000
mean     1500.000000
std       161.426049
min      1091.188150
25%      1382.164463
50%      1501.793175
75%      1596.972369
max      2000.726332
Name: FinalElo, dtype: float64


Highest 10 Elo ratings in 2026:


,Season,TeamID,FinalElo
13638,2026,1181,2000.726332
13540,2026,1112,1969.922449
13558,2026,1276,1931.321250
13436,2026,1385,1896.350654
13464,2026,1438,1872.771206
13590,2026,1211,1866.786795
13541,2026,1196,1852.330736
13492,2026,1433,1841.914533
13538,2026,1116,1834.752983
13584,2026,1222,1823.037438



Lowest 10 Elo ratings in 2026:


...

count     363.000000
mean     1500.000000
std       247.987872
min       868.914695
25%      1316.313193
50%      1501.688059
75%      1674.654220
max      2207.498248
Name: FinalElo, dtype: float64


Highest 10 Elo ratings in 2026:


,Season,TeamID,FinalElo
9556,2026,3417,2207.498248
9702,2026,3163,2153.617900
9566,2026,3400,2140.727364
9490,2026,3376,2104.771182
9540,2026,3452,2045.516600
9603,2026,3181,2023.846015
9658,2026,3234,2006.274951
9694,2026,3261,1989.672760
9740,2026,3395,1982.741366
9827,2026,3343,1952.843008



Lowest 10 Elo ratings in 2026:


,Season,TeamID,FinalElo
9553,2026,3341,868.914695
9705,2026,3434,945.397010
9746,2026,3342,945.477721
9847,2026,3354,965.755897
9845,2026,3441,978.369802
9798,2026,3468,987.422092
9589,2026,3384,1007.465429
9768,2026,3310,1020.786535
9599,2026,3148,1022.489928
9683,2026,3481,1031.866785


In [22]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MEN_ELO_OUTPUT = PROCESSED_DIR / "men_elo.csv"
WOMEN_ELO_OUTPUT = PROCESSED_DIR / "women_elo.csv"

men_elo.to_csv(MEN_ELO_OUTPUT, index=False)
women_elo.to_csv(WOMEN_ELO_OUTPUT, index=False)

print("Saved:", MEN_ELO_OUTPUT)
print("Saved:", WOMEN_ELO_OUTPUT)

Saved: ../data/processed/men_elo.csv
Saved: ../data/processed/women_elo.csv
